<a href="https://colab.research.google.com/github/sow2425/Generative-AI-and-LLM-Learning/blob/main/AI_Text_Image_Question_Answering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers accelerate sentencepiece gradio pillow

In [ ]:
import torch
import gradio as gr

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    BlipProcessor,
    BlipForQuestionAnswering
)

from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Libraries imported successfully!")
print("Running on:", device)

if device == "cuda":
    print(torch.cuda.get_device_name(0))

Libraries imported successfully!
Running on: cpu


In [ ]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

text_model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

text_model = text_model.to(device)

print("FLAN-T5 model loaded successfully!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


FLAN-T5 model loaded successfully!


In [ ]:
def text_question_answer(question):

    if not question.strip():
        return "Please enter a question."

    prompt = f"""
    Answer the following question clearly and correctly.
    Give a simple explanation.

    Question: {question}

    Answer:
    """

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():

        outputs = text_model.generate(
            **inputs,
            max_new_tokens=150,
            num_beams=4
        )

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return answer

In [ ]:
question = "What is Python programming language?"

answer = text_question_answer(question)

print("Question:", question)
print("Answer:", answer)

Question: What is Python programming language?
Answer: Python is a programming language. Python is a programming language. The answer: programming language.


In [ ]:
print(text_question_answer("What is machine learning?"))

print(text_question_answer("Explain object oriented programming."))

print(text_question_answer("What is the difference between AI and ML?"))

A machine learning algorithm is a computer program that learns information from a computer program. The machine learning algorithm is a computer program that learns information from a computer program. The machine learning algorithm is a computer program that learns information from a computer program. The machine learning algorithm is a computer program that learns information from a computer program. The machine learning algorithm is a computer program that learns information from a computer program. The machine learning algorithm is a computer program that learns information from a computer program. The machine learning algorithm is a computer program that learns information from a computer program. The machine learning algorithm is a computer program that learns information from a computer
Object oriented programming is the use of a computer program to program objects. Object oriented programming is the use of a computer program to program objects. The answer: Object oriented progr

In [ ]:
image_model_name = "Salesforce/blip-vqa-base"

processor = BlipProcessor.from_pretrained(
    image_model_name
)

image_model = BlipForQuestionAnswering.from_pretrained(
    image_model_name
)

image_model = image_model.to(device)

print("BLIP image model loaded successfully!")

preprocessor_config.json:   0%|          | 0.00/445 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.54GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/788 [00:00<?, ?it/s]

BLIP image model loaded successfully!


In [ ]:
def image_question_answer(image, question):

    if image is None:
        return "Please upload an image."

    if not question.strip():
        return "Please enter a question."

    image = image.convert("RGB")

    inputs = processor(
        image,
        question,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():

        output = image_model.generate(
            **inputs,
            max_new_tokens=50
        )

    answer = processor.decode(
        output[0],
        skip_special_tokens=True
    )

    return answer

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving WhatsApp Image 2026-11-16 at 4.15.23 PM.jpeg to WhatsApp Image 2026-11-16 at 4.15.23 PM (2).jpeg


In [ ]:
import os

image_path = list(uploaded.keys())[0]

img = Image.open(image_path).convert("RGB")

question = "What is present in this image?"

answer = image_question_answer(img, question)

print("Question:", question)
print("Answer:", answer)

Question: What is present in this image?
Answer: two girls


In [ ]:
question = "What is the color of the main object?"

answer = image_question_answer(img, question)

print("Question:", question)
print("Answer:", answer)

Question: What is the color of the main object?
Answer: blue


In [ ]:
with gr.Blocks(title="AI Question Answering System") as demo:

    gr.Markdown(
        "# AI-Based Question Answering System"
    )

    gr.Markdown(
        "Ask questions using text or upload an image "
        "to get AI-generated answers."
    )

    # MODULE 1: TEXT QUESTION ANSWERING

    with gr.Tab("Text Question Answering"):

        gr.Markdown("## Ask Any Question")

        text_question = gr.Textbox(
            label="Enter Your Question",
            placeholder="Example: What is Artificial Intelligence?"
        )

        text_btn = gr.Button("Get Answer")

        text_result = gr.Textbox(
            label="AI Generated Answer",
            lines=5
        )

        text_btn.click(
            fn=text_question_answer,
            inputs=text_question,
            outputs=text_result
        )

    # MODULE 2: IMAGE QUESTION ANSWERING

    with gr.Tab("Image Question Answering"):

        gr.Markdown("## Upload Image and Ask Questions")

        image_input = gr.Image(
            label="Upload Your Image",
            type="pil"
        )

        image_question = gr.Textbox(
            label="Enter Your Question",
            placeholder="What is present in this image?"
        )

        image_btn = gr.Button("Get Image Answer")

        image_result = gr.Textbox(
            label="AI Generated Answer",
            lines=4
        )

        image_btn.click(
            fn=image_question_answer,
            inputs=[image_input, image_question],
            outputs=image_result
        )

demo.launch(share=True)

NameError: name 'gr' is not defined